# Building a stock-analyzing AI agent, by pairing a LangGraph ReAct agent with a tool that fetches stock market data using yfinance to calculate percentage gains or drops from the previous close.

In [1]:
import yfinance as yf
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

load_dotenv()

True

# Define Tool - identify declining stocks

In [2]:
@tool
def get_bluechip_declining_stocks() -> str:
    """
    Fetches market data for major blue-chip stocks and returns the top 5 
    decliners based on percentage loss from yesterday's market close.
    """
    # Sample list of major blue-chip tickers
    blue_chips = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "JPM", "PG", "JNJ", "V", "BRK-B"]
    
    stock_data = []
    for symbol in blue_chips:
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            
            if len(hist) >= 2:
                prev_close = hist['Close'].iloc[-2]
                latest_close = hist['Close'].iloc[-1]
                pct_change = ((latest_close - prev_close) / prev_close) * 100
                
                stock_data.append({
                    "ticker": symbol,
                    "latest_price": round(latest_close, 2),
                    "change_percent": round(pct_change, 2)
                })
        except Exception:\
            continue
    
    # Sort stocks ascending (lowest/most negative percent change first)
    sorted_losers = sorted(stock_data, key=lambda x: x["change_percent"])
    return str(sorted_losers[:5])

# Define Tool - identify gaining stocks

In [3]:
@tool
def get_bluechip_gaining_stocks() -> str:
    """
    Fetches market data for major blue-chip stocks and returns the top 5 
    gainers based on percentage growth from yesterday's market close.
    Use this tool when users ask about stocks at near-highs or profit-taking opportunities.
    """
    blue_chips = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "JPM", "PG", "JNJ", "V", "BRK-B"]
    
    stock_data = []
    for symbol in blue_chips:
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            
            if len(hist) >= 2:
                prev_close = hist['Close'].iloc[-2]
                latest_close = hist['Close'].iloc[-1]
                pct_change = ((latest_close - prev_close) / prev_close) * 100
                
                stock_data.append({
                    "ticker": symbol,
                    "latest_price": round(latest_close, 2),
                    "change_percent": round(pct_change, 2)
                })
        except Exception:
            continue
    
    # Sort descending to get the highest positive percentage changes first
    sorted_gainers = sorted(stock_data, key=lambda x: x["change_percent"], reverse=True)
    return str(sorted_gainers[:5])

In [4]:
# Step 2: Initialize Gemini LLM
model = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it",
    temperature=0.2
)

# Step 3: Define Agent System Instructions & Build Agent
system_prompt = """You are a smart stock market analysis assistant.
When a user asks to buy blue-chip stocks at a dip:
1. Call the `get_bluechip_declining_stocks` tool to fetch the biggest DECLINERS from yesterday's close.
2. Call the `get_bluechip_gaining_stocks` tool to fetch the biggest GAINERS from yesterday's close.
3. Filter and analyze the top 3-5 biggest gainers/losers returned by the tool.
4. Present your analysis clearly, outlining the ticker, price, percent drop, and a brief technical/fundamental observation.
5. Include a brief disclaimer that this is educational information and not formal financial advice."""

agent = create_agent(
    model=model,
    tools=[get_bluechip_declining_stocks, get_bluechip_gaining_stocks],
    system_prompt=system_prompt  # Use 'system_prompt' instead of 'prompt'
)

In [5]:
# Step 4: Run the Agent Query
user_query = "I want to buy a blue chip stock at the dip, tell me the biggest loser of yesterday's market close."

result = agent.invoke({"messages": [("user", user_query)]})


In [6]:
# PRINT OUTPUT NEATLY

response_blocks = result["messages"][-1].content

if isinstance(response_blocks, list):
    for block in response_blocks:
        # Display internal reasoning block
        if block.get("type") == "thinking":
            print("🤔 THINKING PROCESS:\n")
            print(block.get("thinking").strip())
            print("\n" + "─" * 60 + "\n")
            
        # Display final response to user
        elif block.get("type") == "text":
            print("💬 FINAL RESPONSE:\n")
            print(block.get("text").strip())
else:
    # Standard string fallback
    print(response_blocks)

🤔 THINKING PROCESS:

The user specifically asked for the biggest loser of yesterday's market close to buy at a dip.
From the `get_bluechip_declining_stocks` output:
- AAPL: -7.35% (Price: 308.91)
- V: -0.04% (Price: 366.13)
- JNJ: +0.21% (Not a loser)
- JPM: +0.27% (Not a loser)
- BRK-B: +0.36% (Not a loser)

The biggest loser is AAPL.

I also have the gainers:
- AMZN: +15.32%
- GOOGL: +6.73%
- MSFT: +3.02%
- NVDA: +2.93%
- PG: +0.37%

Now I will present the analysis as requested.

────────────────────────────────────────────────────────────

💬 FINAL RESPONSE:

Based on yesterday's market close, the biggest loser among the major blue-chip stocks is **Apple Inc. (AAPL)**.

### Analysis of Top Decliners
| Ticker | Price | Percent Drop | Observation |
| :--- | :--- | :--- | :--- |
| **AAPL** | $308.91 | **-7.35%** | A significant pullback that may present a "buy the dip" opportunity for long-term investors, depending on the catalyst for the drop. |
| **V** | $366.13 | **-0.04%** | Trading

In [7]:
# Step 4: Run the Agent Query - for gainers
user_query = "I want to sell a blue chip stock at the dip, tell me the biggest gainer of yesterday's market close."

result = agent.invoke({"messages": [("user", user_query)]})

In [8]:
# PRINT OUTPUT NEATLY

response_blocks = result["messages"][-1].content

if isinstance(response_blocks, list):
    for block in response_blocks:
        # Display internal reasoning block
        if block.get("type") == "thinking":
            print("🤔 THINKING PROCESS:\n")
            print(block.get("thinking").strip())
            print("\n" + "─" * 60 + "\n")
            
        # Display final response to user
        elif block.get("type") == "text":
            print("💬 FINAL RESPONSE:\n")
            print(block.get("text").strip())
else:
    # Standard string fallback
    print(response_blocks)

🤔 THINKING PROCESS:

The tool `get_bluechip_gaining_stocks` returned a list of the top 5 gainers.
The biggest gainer is AMZN with a change of 15.32% and a price of 271.58.

The user's prompt was "I want to sell a blue chip stock at the dip, tell me the biggest gainer of yesterday's market close."
Actually, selling at a "dip" is usually not the goal when looking for gainers (you'd sell at a peak), but the user explicitly asked for the biggest gainer.

I will provide the analysis for the top gainers as per the system instructions (though the system instructions specifically mention "buy blue-chip stocks at a dip", the user is asking for gainers). The system instructions say: "When a user asks to buy blue-chip stocks at a dip... Call get_bluechip_declining_stocks... Call get_bluechip_gaining_stocks...". Even though the user said "sell at the dip" (which is confusing phrasing), they asked for the biggest gainer.

I will list the top gainers and highlight the biggest one.

Analysis:
- AMZN: